# 🔐 Notebook 2: CRC32 vs MD5 vs SHA-256

All three are checksums. They differ in **purpose** and **cost**:

- **CRC32** — fast, designed for accidental bit errors (storage, network framing). Not collision-resistant against an adversary.
- **MD5** — cryptographically *broken*, but still useful for accidental-corruption detection; faster than SHA-256.
- **SHA-256** — collision-resistant against active attackers; slowest.


## 🛠️ Setup

```bash
cd 02-distributed-primitives/checksum
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 🛠️ Helpers

In [ ]:
import zlib, hashlib, time

def crc32(b): return zlib.crc32(b).to_bytes(4,'big')
def md5(b):   return hashlib.md5(b).digest()
def sha256(b):return hashlib.sha256(b).digest()

ALGS = {'crc32': crc32, 'md5': md5, 'sha256': sha256}

def store(path, payload, alg):
    digest = ALGS[alg](payload)
    with open(path,'wb') as f:
        f.write(len(digest).to_bytes(2,'big')); f.write(digest); f.write(payload)

def load(path, alg):
    with open(path,'rb') as f:
        n = int.from_bytes(f.read(2),'big')
        stored = f.read(n)
        payload = f.read()
    if ALGS[alg](payload) != stored:
        raise ValueError('checksum mismatch — file is corrupt')
    return payload


## ✅ Round-trip + corruption test

In [ ]:
import os, tempfile
WORKDIR = tempfile.mkdtemp(prefix='chk_')
data = b'hello world ' * 100

for alg in ALGS:
    p = os.path.join(WORKDIR, f'data.{alg}')
    store(p, data, alg)
    print(f'{alg}: roundtrip ok? {load(p, alg) == data}')
    # corrupt one byte and try again
    raw = bytearray(open(p,'rb').read()); raw[-1] ^= 0xFF
    open(p,'wb').write(raw)
    try: load(p, alg); print('  ❌ corruption NOT detected')
    except ValueError as e: print(f'  ✅ detected: {e}')


## ⚡ Performance comparison on 10 MB

In [ ]:
blob = os.urandom(10 * 1024 * 1024)
for name, fn in ALGS.items():
    t0 = time.perf_counter()
    for _ in range(5): fn(blob)
    print(f'{name:7s} {(time.perf_counter()-t0)*1000/5:7.2f} ms / 10MB')


## 📊 Choosing

| Goal | Algorithm |
|---|---|
| Detect random bit flips on disk / wire | **CRC32** (cheap and good enough) |
| Cheap content fingerprint, no adversary | **MD5** (faster than SHA-256) |
| Tamper detection, signatures, dedup at scale | **SHA-256** (or BLAKE3) |

Most real storage layers use **CRC32** per page/segment to catch hardware faults, plus **SHA-256** for end-to-end integrity at the object level.